In [1]:
"""
 NEXO AI Assistant - QUICK RUN (Error-Free Setup):

Dependencies ready in requirements.txt
README.md has full instructions

To run WITHOUT errors:
1. python.org → Install Python + "Add to PATH" ✓
2. Restart PowerShell/VSCode
3. pip install -r requirements.txt
4. python "nexo ai.py" → GUI launches!

Code fully validated: No syntax/import errors.
Ready-to-run post-PATH fix.
"""

import os
import random
import datetime
import webbrowser
import subprocess
import tkinter as tk
from tkinter import ttk, messagebox
import pyttsx3
import speech_recognition as sr
import pyjokes
from PIL import ImageGrab
import ctypes
import psutil

class NexoPersonality:
    """AI personality with mood-based responses"""
    
    def __init__(self):
        self.engine = pyttsx3.init('sapi5')
        voices = self.engine.getProperty('voices')
        if voices:
            self.engine.setProperty('voice', voices[0].id)
        self.engine.setProperty('rate', 180)
        
        self.moods = ['cheerful', 'professional', 'playful', 'wise']
        self.current_mood = random.choice(self.moods)
        self.name = "NEXO"
        
    def speak(self, text, mood=None):
        """Speak with personality"""
        if mood is None:
            mood = self.current_mood
            
        prefixes = {
            'cheerful': "Hey! ",
            'professional': "Certainly. ",
            'playful': "Hehe! ",
            'wise': "As I see it... "
        }
        
        full_text = prefixes.get(mood, "") + text
        print(f"🗣️ {self.name}: {full_text}")
        self.engine.say(full_text)
        self.engine.runAndWait()

class SmartAssistant:
    """Core AI assistant functionality"""
    
    def __init__(self):
        self.personality = NexoPersonality()
        self.recognizer = sr.Recognizer()
        self.user_name = "User"
        
    def listen(self):
        """Advanced voice recognition"""
        with sr.Microphone() as source:
            self.recognizer.adjust_for_ambient_noise(source, duration=0.5)
            self.recognizer.pause_threshold = 1
            
            try:
                self.personality.speak("I'm listening...")
                audio = self.recognizer.listen(source, timeout=5, phrase_time_limit=5)
                
                command = self.recognizer.recognize_google(audio, language='en-in').lower()
                print(f"👂 You said: {command}")
                return command
                
            except sr.UnknownValueError:
                self.personality.speak("I didn't catch that. Could you repeat?", mood='playful')
                return None
            except sr.RequestError:
                self.personality.speak("Connection issue. Let me try again.", mood='professional')
                return None

    def process_command(self, command):
        """Process commands intelligently"""
        if not command:
            return
            
        # Greetings
        greetings = ['hello', 'hi', 'hey', 'good morning', 'good afternoon', 'good evening']
        for greeting in greetings:
            if greeting in command:
                self.personality.speak(f"Hello {self.user_name}! I'm {self.personality.name}, your AI assistant!")
                return
                
        # Commands
        if 'joke' in command:
            self.tell_joke()
        elif 'time' in command:
            self.tell_time()
        elif 'weather' in command:
            self.get_weather()
        elif 'news' in command:
            self.get_news()
        elif 'screenshot' in command:
            self.take_screenshot()
        elif 'music' in command:
            self.play_music()
        elif 'system' in command:
            self.show_system_info()
        elif 'lock' in command:
            self.lock_system()
        elif 'search' in command:
            self.smart_search(command.replace('search', '').strip())
        elif 'open' in command:
            app_name = command.replace('open', '').strip()
            self.open_application(app_name)
        elif 'help' in command:
            self.show_help()
        else:
            self.smart_search(command)

    def tell_joke(self):
        """Tell AI-themed jokes"""
        joke = pyjokes.get_joke()
        self.personality.speak(joke, mood='playful')

    def tell_time(self):
        """Tell time with personality"""
        now = datetime.datetime.now()
        time_str = now.strftime("%I:%M %p")
        day_str = now.strftime("%A, %B %d")
        self.personality.speak(f"It's {time_str} on this beautiful {day_str}!", mood='cheerful')

    def take_screenshot(self):
        """Take smart screenshot"""
        screenshot = ImageGrab.grab()
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"nexo_screenshot_{timestamp}.png"
        
        os.makedirs("nexo_screenshots", exist_ok=True)
        filepath = os.path.join("nexo_screenshots", filename)
        screenshot.save(filepath)
        self.personality.speak(f"Screenshot saved as {filename}!", mood='cheerful')

    def play_music(self):
        """Play music intelligently"""
        music_dir = os.path.expanduser("~/Music")
        if os.path.exists(music_dir):
            music_files = [f for f in os.listdir(music_dir) if f.endswith(('.mp3', '.wav'))]
            if music_files:
                song = random.choice(music_files)
                song_path = os.path.join(music_dir, song)
                os.startfile(song_path)
                self.personality.speak(f"Now playing: {song}", mood='cheerful')
            else:
                self.personality.speak("No music found. Want me to open YouTube?", mood='professional')
        else:
            self.personality.speak("Music folder not found!", mood='wise')

    def get_weather(self):
        """Weather with personality"""
        weather = ["Beautiful sunny day!", "Perfect coding weather!", "Great day to stay indoors!"]
        self.personality.speak(random.choice(weather), mood='cheerful')

    def get_news(self):
        """Open news"""
        webbrowser.open("https://news.google.com")
        self.personality.speak("Opening the latest news!", mood='professional')

    def show_system_info(self):
        """Display system info"""
        cpu = psutil.cpu_percent()
        memory = psutil.virtual_memory()
        self.personality.speak(f"System running great! CPU: {cpu}%, Memory: {memory.percent}%", mood='professional')

    def open_application(self, app_name):
        """Open applications"""
        apps = {
            'chrome': 'chrome', 'notepad': 'notepad', 'calculator': 'calc',
            'paint': 'mspaint', 'cmd': 'cmd', 'explorer': 'explorer'
        }
        
        if app_name in apps:
            try:
                subprocess.Popen(apps[app_name])
                self.personality.speak(f"Opening {app_name}!", mood='cheerful')
            except:
                self.personality.speak(f"Couldn't open {app_name}", mood='professional')
        else:
            self.personality.speak(f"Try: chrome, notepad, calculator, paint, cmd", mood='playful')

    def smart_search(self, query):
        """Smart web search"""
        if query:
            search_url = f"https://www.google.com/search?q={query.replace(' ', '+')}"
            webbrowser.open(search_url)
            self.personality.speak(f"Searching for '{query}'!", mood='professional')

    def lock_system(self):
        """Lock system"""
        self.personality.speak("Locking your system. See you soon!", mood='professional')
        ctypes.windll.user32.LockWorkStation()

    def show_help(self):
        """Show help"""
        help_text = """
🤖 NEXO Commands:
• Hello/Hi - Greet your AI
• Joke - Tell a funny joke
• Time - Current time
• Weather - Weather info
• News - Latest news
• Screenshot - Take screenshot
• Music - Play music
• System - System info
• Search [query] - Web search
• Open [app] - Open apps
• Lock - Lock system
• Help - Show this help
        """
        print(help_text)
        self.personality.speak("Here are all my commands!", mood='cheerful')

class NexoGUI:
    """Modern GUI interface"""
    
    def __init__(self, root):
        self.root = root
        self.assistant = SmartAssistant()
        
        # Window setup
        self.root.title("🤖 NEXO AI Assistant")
        self.root.geometry("700x500")
        self.root.configure(bg='#1e1e1e')
        
        self.create_widgets()
        
    def create_widgets(self):
        """Create modern interface"""
        # Header
        header = tk.Label(self.root, text="🤖 NEXO AI", 
                         font=('Segoe UI', 24, 'bold'),
                         bg='#1e1e1e', fg='#00ff88')
        header.pack(pady=20)

        # Status
        self.status_label = tk.Label(self.root, text="Ready to assist!",
                                     font=('Segoe UI', 12),
                                     bg='#1e1e1e', fg='white')
        self.status_label.pack()

        # Command entry
        self.command_frame = tk.Frame(self.root, bg='#1e1e1e')
        self.command_frame.pack(pady=20)

        self.command_entry = tk.Entry(self.command_frame, width=40,
                                    font=('Segoe UI', 14),
                                    bg='#2d2d2d', fg='white',
                                    insertbackground='white')
        self.command_entry.pack(side=tk.LEFT, padx=10)
        self.command_entry.bind('<Return>', lambda e: self.process_command())

        # Buttons
        button_frame = tk.Frame(self.root, bg='#1e1e1e')
        button_frame.pack(pady=10)

        buttons = [
            ("🎤 Voice", self.voice_command, '#00ff88'),
            ("📋 Help", self.show_help, '#ff6b6b'),
            ("🔒 Lock", self.lock_system, '#ff9500'),
            ("📸 Screenshot", self.take_screenshot, '#00bfff')
        ]

        for text, command, color in buttons:
            btn = tk.Button(button_frame, text=text, command=command,
                          bg=color, fg='white', font=('Segoe UI', 11, 'bold'),
                          relief='flat', padx=15, pady=8)
            btn.pack(side=tk.LEFT, padx=5)

        # Output area
        self.output_text = tk.Text(self.root, height=12, width=60,
                                 bg='#2d2d2d', fg='white',
                                 font=('Consolas', 11))
        self.output_text.pack(pady=20)

    def process_command(self):
        """Process text commands"""
        command = self.command_entry.get()
        if command:
            self.log_output(f"You: {command}")
            self.assistant.process_command(command.lower())
            self.command_entry.delete(0, tk.END)

    def voice_command(self):
        """Handle voice commands"""
        self.status_label.config(text="Listening...")
        self.root.update()
        
        command = self.assistant.listen()
        if command:
            self.log_output(f"Voice: {command}")
            self.assistant.process_command(command)
        
        self.status_label.config(text="Ready!")

    def show_help(self):
        """Show help in GUI"""
        self.assistant.show_help()

    def lock_system(self):
        """Lock system from GUI"""
        self.assistant.lock_system()
        self.log_output("System locked!")

    def take_screenshot(self):
        """Take screenshot from GUI"""
        self.assistant.take_screenshot()
        self.log_output("Screenshot taken!")

    def log_output(self, text):
        """Log output to text area"""
        self.output_text.insert(tk.END, f"{text}\n")
        self.output_text.see(tk.END)

def main():
    """Main application"""
    root = tk.Tk()
    app = NexoGUI(root)
    
    # Welcome
    app.assistant.personality.speak("Welcome to NEXO AI! I'm ready to help!", mood='cheerful')
    
    root.mainloop()

if __name__ == "__main__":
    main()


🗣️ NEXO: Hey! Welcome to NEXO AI! I'm ready to help!
